# 01 — Exploração estatística do NASA C-MAPSS

Este notebook descreve a estrutura dos dados antes de qualquer modelo. Objetivamente queremos saber : **quantas trajetórias existem, se estão completas e como os sensores variam ao longo do tempo**. O conjunto foi criado como uma simulação de degradação de motores até a falha (Saxena et al., 2008). Em termos simples, cada motor é acompanhado ciclo a ciclo para observar como seus sinais mudam antes da falha.

A separação futura será feita por motor, nunca por linhas aleatórias. Essa escolha respeita a organização do conjunto em trajetórias independentes (NASA Ames Prognostics Center of Excellence, 2008). Em termos simples, o modelo será treinado com alguns motores e testado em motores diferentes; isso evita que ele memorize pedaços do mesmo motor nas duas etapas.

## O que significa RUL?

RUL é a sigla de **Remaining Useful Life**, expressão em inglês para **vida útil remanescente**. Neste projeto, ela representa quantos ciclos de operação ainda faltam para um motor atingir a falha. Por exemplo, se um motor está no ciclo 120 e a falha ocorre no ciclo 200, o RUL observado naquele momento é de 80 ciclos.

A previsão de RUL será o principal objetivo da manutenção preditiva aqui estudada. A ideia é estimar esse número com antecedência suficiente para que uma intervenção possa ser planejada antes da falha.

In [15]:
from pathlib import Path
import sys
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'vscode'

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split, sensor_profile, structural_summary
from multilang import LangMap

lang = LangMap(source='pt', target='pt')
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'


## 1. Resumo estrutural

A tabela abaixo é uma verificação de integridade, não uma avaliação de desempenho preditivo.

In [16]:
summary = structural_summary(DATA_DIR)
summary

,subset,dataset,rows,columns,units,cycle_min,cycle_max,missing_values,duplicate_rows,cycles_monotonic_by_unit,terminal_cycle_median
0,train,FD001,20631,26,100,1,362,0,0,True,199.0
1,train,FD002,53759,26,260,1,378,0,0,True,199.0
2,train,FD003,24720,26,100,1,525,0,0,True,220.5
3,train,FD004,61249,26,249,1,543,0,0,True,234.0
4,test,FD001,13096,26,100,1,303,0,0,True,133.5
5,test,FD002,33991,26,259,1,367,0,0,True,132.0
6,test,FD003,16596,26,100,1,475,0,0,True,148.0
7,test,FD004,41214,26,248,1,486,0,0,True,153.5


**Insights:** Os arquivos foram lidos sem valores ausentes ou duplicidades, e a sequência de ciclos foi preservada dentro de cada motor. Isso dá segurança para iniciar a exploração sem uma etapa imediata de correção estrutural. Ainda assim, cada linha pertence a uma trajetória temporal; portanto, as 20.631 observações do FD001 não devem ser tratadas como 20.631 motores independentes. A validação futura deverá ser feita por motor, para evitar que informações da mesma trajetória apareçam simultaneamente no treinamento e na avaliação.

## 2. Sensores constantes e escalas

Uma coluna constante não ajuda a distinguir momentos diferentes. Ela pode ser removida do baseline, mas a decisão deve ser registrada para permanecer auditável.

In [17]:
fd001 = load_split(DATA_DIR, 'train', 'FD001')
profile_fd001 = sensor_profile(fd001)
profile_fd001

,feature,missing,unique_values,minimum,maximum,mean,std
0,s1,0,1,518.6700,518.6700,518.670000,0.000000e+00
1,s10,0,1,1.3000,1.3000,1.300000,0.000000e+00
2,s16,0,1,0.0300,0.0300,0.030000,3.469531e-18
3,s18,0,1,2388.0000,2388.0000,2388.000000,0.000000e+00
4,s19,0,1,100.0000,100.0000,100.000000,0.000000e+00
5,s5,0,1,14.6200,14.6200,14.620000,5.329200e-15
6,setting3,0,1,100.0000,100.0000,100.000000,0.000000e+00
7,s6,0,2,21.6000,21.6100,21.609803,1.388985e-03
8,s17,0,13,388.0000,400.0000,393.210654,1.548763e+00
9,setting2,0,13,-0.0006,0.0006,0.000002,2.930621e-04


**Insights:** Os desvios padrão pequenos devem ser comparados com a escala e a média de cada sensor. Em algumas colunas, como s1, s10, s18, s19 e setting3, a ausência de variação é praticamente total; em outras, como s6, setting1 e setting2, a faixa é estreita, mas ainda existe mudança entre as observações. Modelos preditivos conseguem trabalhar com sinais concentrados, desde que a informação esteja na relação entre a pequena mudança do sensor e a evolução do RUL. Modelos baseados em árvores normalmente não exigem que todas as variáveis estejam na mesma escala, enquanto modelos baseados em distância, margem ou redes neurais costumam se beneficiar de padronização. Por isso, nenhum sensor será descartado apenas por apresentar desvio padrão pequeno: a decisão deverá ser comparada com o desempenho obtido na validação por motor.

In [18]:
constant_fd001 = profile_fd001.loc[profile_fd001['unique_values'] <= 1, 'feature'].tolist()
print('Colunas constantes no FD001:', constant_fd001)

Colunas constantes no FD001: ['s1', 's10', 's16', 's18', 's19', 's5', 'setting3']


**Insights:** No FD001, sete colunas não variam ao longo das observações: s1, s5, s10, s16, s18, s19 e setting3. Como uma coluna sem variação não separa um ciclo de outro neste conjunto, ela poderá ser retirada do primeiro baseline para reduzir complexidade e ruído. Essa decisão é válida para o FD001 e deverá ser reavaliada nos demais conjuntos, pois uma variável constante em uma condição de operação pode variar em outra.

## 3. Comprimento das trajetórias

Cada linha representa um ciclo de operação. A distribuição do número de ciclos por motor mostra quanta experiência temporal existe em cada trajetória.

In [19]:
trajectory_lengths = fd001.groupby('unit', as_index=False)['cycle'].max()
fig = px.histogram(trajectory_lengths, x='cycle', nbins=20,
                   labels={'cycle': 'Ciclos até a falha'},
                   title=lang({'title': 'Comprimento das trajetórias — FD001'})['title'])
fig.update_yaxes(title_text='Número de motores')
fig.show()

**Insights:** Os motores do FD001 não possuem trajetórias do mesmo tamanho. Foram observados comprimentos entre 128 e 362 ciclos, com mediana de 199 ciclos; sete motores terminaram até 150 ciclos e dezessete chegaram a pelo menos 250 ciclos. A diferença é relevante porque um motor com mais registros não representa necessariamente mais motores, mas apenas uma trajetória mais longa. Por isso, médias e métricas deverão ser acompanhadas por uma avaliação por motor e, quando necessário, por um resumo que não seja dominado pelos motores mais longos.

## 4. Exemplo de evolução temporal

O gráfico mostra dois sensores em um motor. Como eles possuem escalas diferentes, cada sensor será apresentado em um painel próprio, com eixo Y independente. A figura é apenas descritiva: ainda não afirma que uma tendência observada será útil para prever RUL em motores novos.

In [20]:
unit_1 = fd001.query('unit == 1')
unit_1_long = unit_1.melt(id_vars='cycle', value_vars=['s3', 's12'],
                        var_name='sensor', value_name='value')
fig = px.line(unit_1_long, x='cycle', y='value', facet_row='sensor',
                labels={'cycle': 'Ciclo de operação', 'value': 'Valor do sensor', 'sensor': 'Sensor'},
                title=lang({'title': 'Evolução de sensores no motor 1 — FD001'})['title'],
                height=700)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.replace('sensor=', '')))
fig.show()

**Insights:** A separação dos painéis permite enxergar melhor a variação de cada sensor, sem que a escala maior de s3 esconda as mudanças de s12. A evolução temporal continua sendo de um único motor e serve para formular hipóteses sobre sinais de degradação, mas não permite afirmar que o mesmo comportamento ocorrerá em motores novos. A utilidade desses sensores para previsão de RUL deverá ser verificada depois, comparando vários motores e separando as trajetórias usadas para treinamento das trajetórias usadas para avaliação.

## Conclusões desta exploração

Os dados são organizados como séries temporais por motor; portanto, a unidade correta de validação é a trajetória completa (Saxena et al., 2008). Em termos simples, o modelo deverá ser avaliado em motores inteiros, e não em linhas misturadas aleatoriamente. Dessa forma, será reduzido o risco de que informações do mesmo motor sejam reutilizadas nas etapas de treinamento e validação.

Na auditoria estrutural não foram encontrados valores ausentes nem linhas duplicadas. No FD001 foram identificadas colunas constantes, que serão excluídas apenas do baseline e permanecerão documentadas para que a decisão possa ser reproduzida.

Nesta etapa não foi medida a acurácia, nenhum modelo foi escolhido e não foi estabelecida uma relação causal entre qualquer sensor e a falha. A exploração foi utilizada somente para compreender a organização, a qualidade e o comportamento inicial dos dados.

A decisão de preservar a separação por trajetória segue a estrutura de séries temporais do conjunto (NASA Ames Prognostics Center of Excellence, 2008) e evita vazamento de informação entre treino e validação.

## Referências

Saxena, A. et al. (2008). *Damage propagation modeling for aircraft engine run-to-failure simulation*. International Conference on Prognostics and Health Management.

NASA Ames Prognostics Center of Excellence (2008). *Turbofan Engine Degradation Simulation Data Set*.
